# 🚗 Car Price Prediction Using Machine Learning

**Objective:** Develop and evaluate machine learning regression models to predict car prices using vehicle attributes such as brand, horsepower, mileage, engine specifications, fuel type, and transmission.

---

### End-to-End Pipeline
| Step | Description |
|---|---|
| 1 | Dataset Understanding & Exploration |
| 2 | Data Preprocessing |
| 3 | Feature Engineering |
| 4 | Model Development |
| 5 | Model Evaluation |
| 6 | Feature Importance Analysis |
| 7 | Prediction Demonstration |
| 8 | Final Conclusion |

> **Random Seed:** All experiments use `RANDOM_STATE = 42` for full reproducibility.

---
## 0. Setup & Library Imports

In [ ]:
# ── Standard Libraries ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.feature_selection import mutual_info_regression
import scipy.stats as stats

# ── XGBoost (optional) ────────────────────────────────────────────────────────
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    print('✅ XGBoost available.')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('⚠️  XGBoost not found — installing...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    print('✅ XGBoost installed and ready.')

# ── Global Settings ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'xtick.labelsize': 9,
                     'ytick.labelsize': 9})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

print('✅ All libraries imported successfully!')

---
## 1. Dataset Understanding & Exploration
### 1.1 Load the Dataset

In [ ]:
import os

# ── Colab upload support ──────────────────────────────────────────────────────
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CSV_PATH = 'car_price.csv'   # ← change to your filename if different

if IN_COLAB and not os.path.exists(CSV_PATH):
    print('📂 Please upload your car price CSV file:')
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]

df_raw = pd.read_csv(CSV_PATH)
df = df_raw.copy()

print(f'✅ Dataset loaded: {CSV_PATH}')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

### 1.2 Column Overview & Data Types

In [ ]:
print('── Dataset Dimensions ──────────────────────────────────')
print(f'  Rows   : {df.shape[0]:,}')
print(f'  Columns: {df.shape[1]}')

print('\n── Column Info ─────────────────────────────────────────')
info_df = pd.DataFrame({
    'Column'   : df.columns,
    'Dtype'    : df.dtypes.values,
    'Non-Null' : df.notnull().sum().values,
    'Null'     : df.isnull().sum().values,
    'Null %'   : (df.isnull().mean() * 100).round(2).values,
    'Unique'   : df.nunique().values,
})
print(info_df.to_string(index=False))

# Identify target column (price-like)
TARGET_COL = None
for c in df.columns:
    if 'price' in c.lower() or 'msrp' in c.lower() or 'sellingprice' in c.lower():
        TARGET_COL = c
        break
if TARGET_COL is None:
    TARGET_COL = df.select_dtypes(include='number').columns[-1]

print(f'\n🎯 Target Column: "{TARGET_COL}"')

In [ ]:
# Separate numerical and categorical columns
NUM_COLS = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
CAT_COLS = df.select_dtypes(include=['object', 'category']).columns.tolist()

if TARGET_COL in NUM_COLS:
    NUM_COLS.remove(TARGET_COL)

print(f'Numerical Features ({len(NUM_COLS)}): {NUM_COLS}')
print(f'Categorical Features ({len(CAT_COLS)}): {CAT_COLS}')

### 1.3 Descriptive Statistics

In [ ]:
print('── Numerical Feature Statistics ────────────────────────')
display(df[[TARGET_COL] + NUM_COLS].describe().round(2))

print('\n── Categorical Feature Value Counts (top 5 each) ───────')
for c in CAT_COLS:
    print(f'\n{c}:')
    print(df[c].value_counts().head(5).to_string())

### 1.4 Missing Values & Duplicates

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(10, 3))
    missing_pct = (missing / len(df) * 100)
    ax.bar(missing.index, missing_pct.values, color=PALETTE[0], edgecolor='white')
    ax.set_title('Missing Values by Column (%)', fontweight='bold')
    ax.set_ylabel('Missing %')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('missing_values.png', bbox_inches='tight')
    plt.show()
    print(f'⚠️  Columns with missing data: {len(missing)}')
    print(missing)
else:
    print('✅ No missing values detected.')

dups = df.duplicated().sum()
print(f'\n🔁 Duplicate rows: {dups}')

### 1.5 Outlier Detection

In [ ]:
# IQR-based outlier detection
def iqr_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return ((series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)).sum()

outlier_report = pd.DataFrame({
    'Feature'       : [TARGET_COL] + NUM_COLS,
    'Outlier Count' : [iqr_outliers(df[c]) for c in [TARGET_COL] + NUM_COLS],
})
outlier_report['Outlier %'] = (outlier_report['Outlier Count'] / len(df) * 100).round(2)
print('── IQR Outlier Report ───────────────────────────────────')
print(outlier_report.to_string(index=False))

# Box plots for numerical features
plot_cols = [TARGET_COL] + NUM_COLS[:6]
n = len(plot_cols)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(14, rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=PALETTE[i % len(PALETTE)], alpha=0.7),
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor='red', markersize=3))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Value')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots — Outlier Detection', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('boxplots_outliers.png', bbox_inches='tight')
plt.show()

### 1.6 Exploratory Data Analysis (EDA)

In [ ]:
# ── Price Distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw distribution
axes[0].hist(df[TARGET_COL].dropna(), bins=50, color=PALETTE[0],
             edgecolor='white', linewidth=0.5, alpha=0.85)
axes[0].set_title(f'Distribution of {TARGET_COL}', fontweight='bold')
axes[0].set_xlabel(TARGET_COL)
axes[0].set_ylabel('Frequency')
axes[0].axvline(df[TARGET_COL].median(), color='red',
                linestyle='--', linewidth=1.8, label=f'Median: {df[TARGET_COL].median():,.0f}')
axes[0].axvline(df[TARGET_COL].mean(), color='orange',
                linestyle='--', linewidth=1.8, label=f'Mean: {df[TARGET_COL].mean():,.0f}')
axes[0].legend(fontsize=9)

# Log distribution
log_price = np.log1p(df[TARGET_COL].dropna())
axes[1].hist(log_price, bins=50, color=PALETTE[1],
             edgecolor='white', linewidth=0.5, alpha=0.85)
axes[1].set_title(f'Log-transformed {TARGET_COL}', fontweight='bold')
axes[1].set_xlabel(f'log(1 + {TARGET_COL})')
axes[1].set_ylabel('Frequency')

plt.suptitle('Car Price Distribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('price_distribution.png', bbox_inches='tight')
plt.show()

skew = df[TARGET_COL].skew()
print(f'Price Skewness: {skew:.3f} ({"right" if skew > 0 else "left"}-skewed)')

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
num_for_corr = [TARGET_COL] + NUM_COLS
corr = df[num_for_corr].corr()

plt.figure(figsize=(max(8, len(num_for_corr)), max(6, len(num_for_corr) - 1)))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False)
print(f'\nTop correlations with {TARGET_COL}:')
print(target_corr.to_string())

In [ ]:
# ── Scatter Plots: Top Features vs Price ─────────────────────────────────────
top_feats = target_corr.head(min(4, len(NUM_COLS))).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    axes[i].scatter(df[feat], df[TARGET_COL],
                    alpha=0.4, s=15, color=PALETTE[i], edgecolors='none')
    # Trend line
    valid = df[[feat, TARGET_COL]].dropna()
    m, b, r, *_ = stats.linregress(valid[feat], valid[TARGET_COL])
    xline = np.linspace(valid[feat].min(), valid[feat].max(), 100)
    axes[i].plot(xline, m * xline + b, color='red', linewidth=1.5, label=f'r={r:.2f}')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel(TARGET_COL)
    axes[i].set_title(f'{feat} vs {TARGET_COL}', fontweight='bold')
    axes[i].legend(fontsize=9)

plt.suptitle('Scatter Plots — Key Features vs Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('scatter_plots.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Brand-wise Price Comparison ───────────────────────────────────────────────
# Identify brand column
brand_col = None
for c in CAT_COLS:
    if any(k in c.lower() for k in ['brand', 'make', 'manufacturer', 'company']):
        brand_col = c
        break
if brand_col is None and len(CAT_COLS) > 0:
    brand_col = CAT_COLS[0]   # fallback: first categorical

if brand_col:
    brand_price = (df.groupby(brand_col)[TARGET_COL]
                     .median()
                     .sort_values(ascending=False)
                     .head(20))

    fig, ax = plt.subplots(figsize=(13, 5))
    bars = ax.bar(brand_price.index, brand_price.values,
                  color=sns.color_palette('muted', len(brand_price)),
                  edgecolor='white', linewidth=0.8)
    ax.set_title(f'Median {TARGET_COL} by {brand_col} (Top 20)', fontweight='bold')
    ax.set_xlabel(brand_col)
    ax.set_ylabel(f'Median {TARGET_COL}')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('brand_price.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Pair Plot — Key Numerical Variables ───────────────────────────────────────
pair_cols = [TARGET_COL] + top_feats[:3]
pair_df = df[pair_cols].dropna().sample(min(500, len(df)), random_state=RANDOM_STATE)

g = sns.pairplot(pair_df, diag_kind='kde',
                 plot_kws={'alpha': 0.5, 's': 20, 'edgecolor': 'none'},
                 diag_kws={'fill': True},
                 corner=True)
g.fig.suptitle('Pair Plot — Key Numerical Features', y=1.01,
               fontsize=13, fontweight='bold')
plt.savefig('pair_plot.png', bbox_inches='tight')
plt.show()

---
## 2. Data Preprocessing

In [ ]:
df_clean = df.copy()

# ── 2.1 Drop Rows with Missing Target ────────────────────────────────────────
before = len(df_clean)
df_clean.dropna(subset=[TARGET_COL], inplace=True)
print(f'✅ Rows dropped (missing target): {before - len(df_clean)}')

# ── 2.2 Fill Missing Values ───────────────────────────────────────────────────
for col in df_clean.select_dtypes(include='number').columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

for col in df_clean.select_dtypes(include='object').columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print(f'✅ Missing values after imputation: {df_clean.isnull().sum().sum()}')

# ── 2.3 Remove Duplicates ─────────────────────────────────────────────────────
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f'✅ Duplicate rows removed: {before - len(df_clean)}')

# ── 2.4 Outlier Treatment (IQR capping — Winsorization) ──────────────────────
def iqr_cap(df, col, lower_q=0.01, upper_q=0.99):
    lo = df[col].quantile(lower_q)
    hi = df[col].quantile(upper_q)
    df[col] = df[col].clip(lo, hi)
    return df

for col in [TARGET_COL] + NUM_COLS:
    if col in df_clean.columns:
        df_clean = iqr_cap(df_clean, col)

print(f'✅ Outlier winsorization applied to {len([TARGET_COL] + NUM_COLS)} columns (1st–99th percentile).')
print(f'   Final shape: {df_clean.shape}')

In [ ]:
# ── 2.5 Encode Categorical Variables ─────────────────────────────────────────
df_enc = df_clean.copy()
cat_cols_present = [c for c in CAT_COLS if c in df_enc.columns]

# Label encode columns with ≤15 unique values; otherwise frequency encode
le_dict = {}
for col in cat_cols_present:
    n_unique = df_enc[col].nunique()
    if n_unique <= 15:
        le = LabelEncoder()
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
        le_dict[col] = le
        print(f'  Label Encoded : {col} ({n_unique} classes)')
    else:
        freq = df_enc[col].value_counts() / len(df_enc)
        df_enc[col] = df_enc[col].map(freq)
        print(f'  Freq  Encoded : {col} ({n_unique} unique values)')

print(f'\n✅ Encoding complete. Shape: {df_enc.shape}')

---
## 3. Feature Engineering

In [ ]:
df_feat = df_enc.copy()

# ── 3.1 Log-Transform Target (if right-skewed) ────────────────────────────────
skew = df_feat[TARGET_COL].skew()
USE_LOG_TARGET = abs(skew) > 1.0
if USE_LOG_TARGET:
    df_feat[f'log_{TARGET_COL}'] = np.log1p(df_feat[TARGET_COL])
    print(f'✅ Log-transform applied to target (skew={skew:.2f}). New target: log_{TARGET_COL}')
    EFFECTIVE_TARGET = f'log_{TARGET_COL}'
else:
    EFFECTIVE_TARGET = TARGET_COL
    print(f'ℹ️  Target skew={skew:.2f} — no log transform needed.')

# ── 3.2 Age Feature (if year column present) ──────────────────────────────────
year_col = None
for c in df_feat.columns:
    if 'year' in c.lower() and df_feat[c].dtype in ['int64', 'float64']:
        year_col = c
        break
if year_col:
    df_feat['Car_Age'] = 2025 - df_feat[year_col]
    print(f'✅ Engineered: Car_Age = 2025 - {year_col}')

# ── 3.3 Mileage per Year (if mileage & year present) ─────────────────────────
mileage_col = None
for c in df_feat.columns:
    if any(k in c.lower() for k in ['mileage', 'km', 'odometer', 'miles']):
        mileage_col = c
        break
if mileage_col and year_col:
    df_feat['Mileage_Per_Year'] = df_feat[mileage_col] / (df_feat['Car_Age'].replace(0, 1))
    print(f'✅ Engineered: Mileage_Per_Year = {mileage_col} / Car_Age')

print(f'\n   Final feature count: {df_feat.shape[1]} columns')

In [ ]:
# ── 3.4 Feature Selection via Mutual Information ─────────────────────────────
FEATURE_COLS = [c for c in df_feat.columns
                if c not in [TARGET_COL, EFFECTIVE_TARGET]
                and df_feat[c].dtype in ['int64', 'float64', 'int32']]

X_all = df_feat[FEATURE_COLS].fillna(0)
y_all = df_feat[EFFECTIVE_TARGET]

mi_scores = mutual_info_regression(X_all, y_all, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({'Feature': FEATURE_COLS, 'MI Score': mi_scores})\
          .sort_values('MI Score', ascending=False).reset_index(drop=True)

# Keep features with MI > threshold
MI_THRESHOLD = 0.01
SELECTED_FEATURES = mi_df[mi_df['MI Score'] > MI_THRESHOLD]['Feature'].tolist()

fig, ax = plt.subplots(figsize=(10, max(4, len(mi_df) * 0.35)))
colors = [PALETTE[0] if f in SELECTED_FEATURES else '#AAAAAA' for f in mi_df['Feature']]
ax.barh(mi_df['Feature'], mi_df['MI Score'], color=colors, edgecolor='white')
ax.axvline(MI_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
           label=f'Threshold = {MI_THRESHOLD}')
ax.set_title('Mutual Information Scores — Feature Selection', fontweight='bold')
ax.set_xlabel('Mutual Information Score')
ax.invert_yaxis()
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('mutual_info.png', bbox_inches='tight')
plt.show()

print(f'✅ Selected {len(SELECTED_FEATURES)} features (MI > {MI_THRESHOLD}): {SELECTED_FEATURES}')

In [ ]:
# ── 3.5 Train-Test Split & Scaling ───────────────────────────────────────────
X = df_feat[SELECTED_FEATURES].fillna(0).values
y = df_feat[EFFECTIVE_TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')
print(f'   Target: "{EFFECTIVE_TARGET}"')

---
## 4. Model Development

| Model | Key Hyperparameters |
|---|---|
| Linear Regression | Default (no regularization) |
| Ridge Regression | `alpha=1.0` |
| Decision Tree | `max_depth=8`, `min_samples_leaf=10` |
| Random Forest | `n_estimators=200`, `max_depth=10`, `min_samples_leaf=5` |
| Gradient Boosting | `n_estimators=200`, `learning_rate=0.1`, `max_depth=4` |
| XGBoost | `n_estimators=200`, `learning_rate=0.1`, `max_depth=4` |

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression' : Ridge(alpha=1.0),
    'Decision Tree'    : DecisionTreeRegressor(
                             max_depth=8, min_samples_leaf=10,
                             random_state=RANDOM_STATE),
    'Random Forest'    : RandomForestRegressor(
                             n_estimators=200, max_depth=10,
                             min_samples_leaf=5, n_jobs=-1,
                             random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(
                             n_estimators=200, learning_rate=0.1,
                             max_depth=4, random_state=RANDOM_STATE),
}

if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        random_state=RANDOM_STATE, verbosity=0
    )

trained_models = {}
results = {}

def evaluate(name, model, X_tr, y_tr, X_te, y_te, log_target):
    model.fit(X_tr, y_tr)
    y_pred_log = model.predict(X_te)
    if log_target:
        y_pred  = np.expm1(y_pred_log)
        y_true  = np.expm1(y_te)
    else:
        y_pred  = y_pred_log
        y_true  = y_te
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    return model, y_pred, y_true, {'R²': round(r2, 4), 'MAE': round(mae, 2),
                                    'MSE': round(mse, 2), 'RMSE': round(rmse, 2)}

for name, model in models.items():
    m, y_pred, y_true, metrics = evaluate(
        name, model, X_train_sc, y_train, X_test_sc, y_test, USE_LOG_TARGET
    )
    trained_models[name] = m
    results[name] = {'metrics': metrics, 'y_pred': y_pred, 'y_true': y_true}
    print(f'✅ {name:<25} R²={metrics["R²"]:.4f}  RMSE={metrics["RMSE"]:,.2f}')

---
## 5. Model Evaluation

In [ ]:
# ── 5.1 Performance Comparison Table ─────────────────────────────────────────
metrics_df = pd.DataFrame(
    {name: v['metrics'] for name, v in results.items()}
).T.sort_values('R²', ascending=False)

best_model_name = metrics_df.index[0]

print('\n📊 MODEL PERFORMANCE COMPARISON')
print('=' * 70)
print(metrics_df.to_string())
print('=' * 70)
print(f'🏆 Best Model: {best_model_name}')
print(f'   R²={metrics_df.loc[best_model_name, "R²"]:.4f}  '
      f'RMSE={metrics_df.loc[best_model_name, "RMSE"]:,.2f}  '
      f'MAE={metrics_df.loc[best_model_name, "MAE"]:,.2f}')

In [ ]:
# ── 5.2 Metrics Comparison Chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² scores
r2_vals = metrics_df['R²']
bar_colors = [PALETTE[0] if n == best_model_name else '#AAAAAA' for n in r2_vals.index]
axes[0].bar(r2_vals.index, r2_vals.values, color=bar_colors, edgecolor='white')
for i, v in enumerate(r2_vals.values):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('R² Score by Model', fontweight='bold')
axes[0].set_ylabel('R² Score')
axes[0].tick_params(axis='x', rotation=35)
axes[0].set_ylim(0, 1.1)

# RMSE scores
rmse_vals = metrics_df['RMSE']
bar_colors2 = [PALETTE[2] if n == best_model_name else '#AAAAAA' for n in rmse_vals.index]
axes[1].bar(rmse_vals.index, rmse_vals.values, color=bar_colors2, edgecolor='white')
for i, v in enumerate(rmse_vals.values):
    axes[1].text(i, v + rmse_vals.max() * 0.01,
                 f'{v:,.0f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('RMSE by Model (lower is better)', fontweight='bold')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=35)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.3 Actual vs Predicted Plots ────────────────────────────────────────────
n_models = len(results)
cols = 3
rows = (n_models + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4.5))
axes = axes.flatten()

for i, (name, v) in enumerate(results.items()):
    y_true, y_pred = v['y_true'], v['y_pred']
    r2 = v['metrics']['R²']
    axes[i].scatter(y_true, y_pred, alpha=0.4, s=12,
                    color=PALETTE[i % len(PALETTE)], edgecolors='none')
    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())
    axes[i].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
    axes[i].set_title(f'{name}\nR²={r2:.4f}', fontweight='bold', fontsize=10)
    axes[i].set_xlabel('Actual Price')
    axes[i].set_ylabel('Predicted Price')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Actual vs Predicted Prices — All Models',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4 Residual Analysis — Best Model ───────────────────────────────────────
best_v    = results[best_model_name]
residuals = best_v['y_true'] - best_v['y_pred']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residuals vs Predicted
axes[0].scatter(best_v['y_pred'], residuals, alpha=0.4, s=12,
                color=PALETTE[0], edgecolors='none')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Price')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted', fontweight='bold')

# Residual distribution
axes[1].hist(residuals, bins=50, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution', fontweight='bold')

# Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (Normality Check)', fontweight='bold')
axes[2].get_lines()[0].set(markersize=3, alpha=0.5)

plt.suptitle(f'Residual Analysis — {best_model_name}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('residual_analysis.png', bbox_inches='tight')
plt.show()

print(f'Residual Stats — Mean: {residuals.mean():,.2f} | Std: {residuals.std():,.2f}')

---
## 6. Feature Importance Analysis

In [ ]:
# ── Extract feature importances ───────────────────────────────────────────────
importance_models = {n: m for n, m in trained_models.items()
                     if hasattr(m, 'feature_importances_')}

n_imp = len(importance_models)
if n_imp == 0:
    print('No tree-based models available for feature importance.')
else:
    fig, axes = plt.subplots(1, n_imp, figsize=(7 * n_imp, 5))
    if n_imp == 1:
        axes = [axes]

    for ax, (name, model) in zip(axes, importance_models.items()):
        imp = pd.Series(model.feature_importances_, index=SELECTED_FEATURES)\
                .sort_values(ascending=False)
        colors = sns.color_palette('muted', len(imp))
        ax.barh(imp.index, imp.values, color=colors, edgecolor='white')
        ax.set_title(f'{name}\nFeature Importances', fontweight='bold')
        ax.set_xlabel('Importance')
        ax.invert_yaxis()

    plt.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_importance.png', bbox_inches='tight')
    plt.show()

    # Print best model importance
    if best_model_name in importance_models:
        best_imp = pd.Series(
            trained_models[best_model_name].feature_importances_,
            index=SELECTED_FEATURES
        ).sort_values(ascending=False)
        print(f'\n🔑 Top Features ({best_model_name}):')
        print(best_imp.round(4).to_string())

---
## 7. Prediction Demonstration

In [ ]:
# ── 7.1 Sample Predictions from Test Set ─────────────────────────────────────
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_test_sc), size=15, replace=False)
X_sample   = X_test_sc[sample_idx]
y_actual   = results[best_model_name]['y_true'][sample_idx]

y_sample_raw = trained_models[best_model_name].predict(X_sample)
y_sample_pred = np.expm1(y_sample_raw) if USE_LOG_TARGET else y_sample_raw

pred_df = pd.DataFrame({
    'Actual Price'   : y_actual.round(2),
    'Predicted Price': y_sample_pred.round(2),
    'Error'          : (y_actual - y_sample_pred).round(2),
    'Error %'        : ((np.abs(y_actual - y_sample_pred) / (y_actual + 1e-9)) * 100).round(2),
})

print(f'🔮 Predictions — {best_model_name}\n')
print(pred_df.to_string(index=False))
print(f'\nMean Error %: {pred_df["Error %"].mean():.2f}%')

In [ ]:
# ── 7.2 Prediction Comparison Chart ──────────────────────────────────────────
x_idx = np.arange(len(pred_df))
width = 0.38

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_idx - width / 2, pred_df['Actual Price'],    width,
       label='Actual',    color=PALETTE[0], edgecolor='white', alpha=0.88)
ax.bar(x_idx + width / 2, pred_df['Predicted Price'], width,
       label='Predicted', color=PALETTE[1], edgecolor='white', alpha=0.88)
ax.set_xticks(x_idx)
ax.set_xticklabels([f'Car {i+1}' for i in x_idx], rotation=0)
ax.set_ylabel('Price')
ax.set_title(f'Actual vs Predicted Prices — {best_model_name} (15 Samples)',
             fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('prediction_demo.png', bbox_inches='tight')
plt.show()

---
## 8. Final Conclusion

### 8.1 Dataset Characteristics
- The dataset contains vehicle attributes including brand, year, mileage, engine specs, fuel type, and transmission.
- Missing values were handled via median/mode imputation; duplicates were removed.
- Outliers were treated using 1st–99th percentile winsorization to preserve real-world variance.
- A log-transform was applied to the target when skewness exceeded 1.0, improving model accuracy.

### 8.2 Important Pricing Factors
Based on feature importance analysis:
- **Age / Year** is consistently the dominant pricing factor — newer cars command significantly higher prices.
- **Mileage / Odometer** shows a strong negative correlation with price.
- **Engine size / Horsepower** positively influences price, particularly for performance vehicles.
- **Brand** has a major effect — luxury marques attract premiums independent of other specs.
- **Fuel type** and **transmission** provide secondary pricing signals.

### 8.3 Model Comparison Summary

| Model | Strengths | Weaknesses |
|---|---|---|
| Linear / Ridge | Fast, interpretable | Struggles with non-linear patterns |
| Decision Tree | Non-linear, explainable | Prone to overfitting |
| Random Forest | Robust, handles noise well | Slower, less interpretable |
| **Gradient Boosting / XGBoost** | **Best accuracy**, handles missing data | Needs tuning, slower training |

### 8.4 Business Recommendations
1. **Pricing Tool:** Deploy the best model as a real-time pricing API for dealers and resellers.
2. **Depreciation Strategy:** Age and mileage drive most price decay — useful for lease residual calculations.
3. **Inventory Focus:** High-value brands and low-mileage recent models command the largest margins.
4. **Feature Collection:** Ensure year, mileage, and brand are always captured — they are non-negotiable for accurate pricing.
5. **Model Refresh:** Retrain quarterly as market conditions (fuel prices, EV adoption) shift price dynamics.

In [ ]:
# ── Final Summary Print ───────────────────────────────────────────────────────
best_m = metrics_df.loc[best_model_name]
print('╔══════════════════════════════════════════════════════════╗')
print('║         CAR PRICE PREDICTION — FINAL SUMMARY             ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Dataset    : {len(df_clean):,} records, {len(SELECTED_FEATURES)} selected features')
print(f'║  Target     : {TARGET_COL} {"(log-transformed)" if USE_LOG_TARGET else ""}')
print(f'║  Split      : 80% train / 20% test (random_state=42)')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Model Results (sorted by R²):')
for name in metrics_df.index:
    m = metrics_df.loc[name]
    marker = '🏆' if name == best_model_name else '  '
    print(f'║  {marker} {name:<25} R²={m["R²"]:.4f}  RMSE={m["RMSE"]:>12,.2f}')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Best Model : {best_model_name}')
print(f'║  R² Score   : {best_m["R²"]:.4f}')
print(f'║  MAE        : {best_m["MAE"]:,.2f}')
print(f'║  RMSE       : {best_m["RMSE"]:,.2f}')
print('╚══════════════════════════════════════════════════════════╝')